# Notebook D — RL at Inference (TTA): the headline results

## Setup (carried over from your already-run Notebook A + Notebook B)
Same setup cells as Notebook C, but Stage B is skipped entirely — TTA adapts the
policy around the **frozen** backbone, not a Stage-B-adjusted one.

**Fix applied here that wasn't in the original spec:** the in-domain TTA cell
(CELL D3) asserts against a variable `frozen_rmse` that was never actually defined
anywhere in the original notebook — it would have crashed with a `NameError` the
first time you ran it. A small cell computing it independently (same frozen-policy
math, done once, before the TTA loop) has been added below, labeled
**CELL D-FROZEN-CHECK**.

## Setup (from your already-run Notebook A)

In [ ]:
import torch, sys, subprocess
print("torch:", torch.__version__, "| python:", sys.version.split()[0], "| cuda:", torch.cuda.is_available())
TORCH = torch.__version__.split("+")[0]
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))
pip("torch-geometric==2.6.1", "torch-scatter", "torch-sparse", "torch-cluster",
    "-f", f"https://data.pyg.org/whl/torch-{TORCH}+cpu.html")
pip("pyyaml", "scipy", "pandas", "matplotlib", "seaborn")
print("deps OK")

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/cosmic-net"
REPO_URL = "https://github.com/Rusheel86/cosmic-net.git"

# Read GitHub PAT from Kaggle Secrets
# If using kaggle_secrets:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_PAT = user_secrets.get_secret("GITHUB_PAT")

# Clone only if repo doesn't already exist
if not os.path.exists(REPO_DIR):
    auth_url = REPO_URL.replace(
        "https://",
        f"https://x-access-token:{GITHUB_PAT}@"
    )

    # SECURITY: never let the token reach the notebook output. A failed
    # subprocess.check_call raises CalledProcessError whose message embeds the
    # full command line (i.e. the token) — re-raise a sanitized error instead.
    try:
        subprocess.check_call([
            "git", "clone",
            auth_url,
            REPO_DIR
        ])
    except subprocess.CalledProcessError:
        raise RuntimeError(
            f"git clone failed for {REPO_URL} (token redacted) — check the PAT "
            "secret, the repo URL, and Kaggle internet access."
        ) from None
    # The clone writes the token into .git/config (remote origin URL) — remove it.
    subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])

os.chdir(REPO_DIR)
print(os.listdir("."))

In [ ]:
# Mount uploaded data (repo already cloned privately in the previous cell)
import subprocess, os, shutil
os.chdir("/kaggle/working/cosmic-net")
print(os.listdir("."))
# Upload tng100_clustered.csv + best_model_augmented.pt as a Kaggle dataset named "cosmicnet-data"
INPUT = "/kaggle/input/datasets/nealsalian/cosmicnet-data"
os.makedirs("data/raw", exist_ok=True)
shutil.copy(f"{INPUT}/tng100_clustered.csv", "data/raw/tng100_clustered.csv")
os.makedirs("kaggle", exist_ok=True)
if os.path.exists(f"{INPUT}/best_model_augmented.pt"):
    shutil.copy(f"{INPUT}/best_model_augmented.pt", "kaggle/best_model_augmented.pt")
print("data staged:", os.path.getsize("data/raw/tng100_clustered.csv")/1e6, "MB")


In [ ]:
#Load config, force CPU-safe worker settings
import sys, yaml, torch, numpy as np
sys.path.insert(0, ".")
with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["source"] = "tng"
cfg["data"]["num_workers"] = 0         
cfg["data"]["batch_size"] = 16
cfg["model"]["mc_samples"] = 30
cfg["rls"] = {
    "policy_hidden": 64, "lr": 0.001, "entropy_coef": 0.01, "value_coef": 0.5,
    "epochs": 60, "batch_size": 32,
    "target_sparsity_start": 0.9, "target_sparsity_end": 0.4,
    "sparsity_anneal_epochs": 40,
    "w_acc": 1.0, "w_sp": 0.5, "w_conn": 1.0, "w_virial": 1.0, "w_unc": 0.5,
    "virial_anneal_start_epoch": 10, "min_keep_frac": 0.1, "seed": 42,
    # TTA ("RL at inference") — tune on VAL split only, then freeze
    "tta_lr": 1e-4, "tta_steps": 10, "tta_mc_samples": 15, "tta_patience": 3,
    "tta_target_sparsity": 0.5,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| source:", cfg["data"]["source"])

In [ ]:
#Load halos, build graphs, split
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder, build_dataloaders
loader = get_loader(cfg)
halos = loader.load()
print("total halos:", len(halos), "| split", loader.split_data.__name__ if False else "")
train_halos, val_halos, test_halos = loader.split_data()
print(f"train={len(train_halos)} val={len(val_halos)} test={len(test_halos)}")
torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])   # AFTER split_data (it resets RNG)
train_loader, val_loader, test_loader = build_dataloaders(cfg, train_halos, val_halos, test_halos)
b0 = next(iter(test_loader))
print("graph:", b0.x.shape, b0.edge_index.shape, b0.edge_attr.shape, "y:", b0.y.shape)

In [ ]:
# Load frozen backbone and verify it predicts
from model.model import load_model
import torch
import torch_geometric.data as pyg_data

ckpt = "/kaggle/input/datasets/nealsalian/cosmicnet-data/best_model_augmented.pt"

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------
gnn = load_model(ckpt, cfg, device)
gnn.eval()

model_device = next(gnn.parameters()).device

# ---------------------------------------------------------
# Create a valid test graph
# Use the complete graph instead of taking 2 nodes
# with arbitrary edges.
# ---------------------------------------------------------
single = pyg_data.Data(
    x=b0.x,
    edge_index=b0.edge_index,
    edge_attr=b0.edge_attr
)

# One graph containing all nodes
single.batch = torch.zeros(
    single.x.shape[0],
    dtype=torch.long
)

# Move graph to the same device as the model
single = single.to(model_device)

# ---------------------------------------------------------
# Forward pass
# ---------------------------------------------------------
with torch.no_grad():
    pred, _ = gnn(single)

print(
    "smoke prediction:", pred.item(),
    "| device:", model_device,
    "| params:", sum(p.numel() for p in gnn.parameters())
)

In [ ]:
# Inline RL helpers (policy net, sparsify, reward) — matches rls/ package
# PREFER: from rls.policy import EdgePolicyNet
# PREFER: from rls.sparsify import hard_mask, repair_connectivity
# PREFER: from rls.policy_gradient import bernoulli_logp
# (swap in the above if rls/ is importable — see TIP at the top of this file)
import torch
import torch.nn as nn
import torch.nn.functional as F

class EdgePolicyNet(nn.Module):
    def __init__(self, edge_dim=5, node_emb_dim=128, hidden_dim=64):
        super().__init__()
        self.node_proj = nn.Linear(node_emb_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)
        self.fc = nn.Sequential(nn.Linear(hidden_dim*4, hidden_dim), nn.LeakyReLU(0.1),
                                nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(0.1),
                                nn.Linear(hidden_dim, 1))
    def forward(self, edge_attr, node_emb, edge_index, context):
        e = F.leaky_relu(self.edge_proj(edge_attr), 0.1)
        u, v = edge_index
        nu = F.leaky_relu(self.node_proj(node_emb[u]), 0.1)
        nv = F.leaky_relu(self.node_proj(node_emb[v]), 0.1)
        c = context.unsqueeze(0).expand(e.size(0), -1)
        return self.fc(torch.cat([nu, nv, e, c], dim=-1))

def hard_mask(probs, min_keep_frac=0.1):
    # BOOL mask: edge_index[:, mask] / edge_attr[mask] require bool; an int
    # 0/1 mask silently positional-indexes (duplicates edges 0 and 1).
    mask = (probs >= 0.5)
    k = int(torch.ceil(torch.tensor(min_keep_frac) * probs.numel()))
    if mask.sum() < k:
        mask = torch.zeros_like(mask, dtype=torch.bool)
        mask[torch.topk(probs, k).indices] = True
    return mask

def apply_min_keep_floor(mask, probs, min_keep_frac=0.1):
    """Same floor-enforcement as hard_mask, but the candidate mask need
    not be a thresholded-probs mask — e.g. a sampled Bernoulli action.
    If the candidate keeps fewer than min_keep_frac of edges, replace it
    with the top-(min_keep_frac) highest-probability edges instead."""
    assert mask.dtype == torch.bool, f"expected bool mask, got {mask.dtype}"
    mask = mask.clone()
    k_min = int(torch.ceil(torch.tensor(min_keep_frac) * probs.numel()))
    if mask.sum() < k_min:
        top = torch.topk(probs, k_min).indices
        mask = torch.zeros_like(mask, dtype=torch.bool)
        mask[top] = True
    return mask

def repair_connectivity(edge_index, mask):
    mask = mask.clone()
    kept = mask.bool()
    incident = torch.zeros(edge_index.max().item()+1, dtype=torch.long)
    for i in range(edge_index.shape[1]):
        if kept[i]:
            incident[edge_index[0, i]] += 1; incident[edge_index[1, i]] += 1
    for node in (incident == 0).nonzero(as_tuple=True)[0].tolist():
        cand = (edge_index == node).sum(dim=0).bool()
        if cand.any():
            mask[cand.nonzero(as_tuple=True)[0][0]] = True
    return mask

def bernoulli_logp(p, action, eps=1e-8):
    return torch.where(action.bool(), torch.log(p.clamp(eps, 1.0)),
                       torch.log((1-p).clamp(eps, 1.0)))

def virial_ratio_pruned(ke, pe, eps=1e-8):
    return (2.0 * ke) / torch.abs(pe).clamp(min=eps)

## Precompute embeddings + RL training helpers (no training loop run here)

In [ ]:
# CELL 6: Precompute frozen node embeddings + graph contexts for ALL graphs
from torch_geometric.nn import global_mean_pool
import torch_geometric.data as pg
import pandas as pd   # used by the training-log save in Cell 10

gnn.eval()
def prepare(loader):
    out = []
    with torch.no_grad():
        for b in loader:
            b = b.to(device)
            emb = gnn.get_embeddings(b, embedding_point="pre_pooling")   # [N, out]
            ctx = global_mean_pool(emb, b.batch)                          # [B, out]
            for i in range(b.num_graphs):
                g = b.get_example(i)
                n = g.x.shape[0]
                out.append({
                    "x": g.x, "edge_index": g.edge_index, "edge_attr": g.edge_attr,
                    "y": g.y, "ctx": ctx[i], "emb": emb[b.batch == i],
                    "stellar_mass": g.stellar_mass if hasattr(g, "stellar_mass") else torch.ones(n)*1e10,
                    "vel_disp": g.vel_disp if hasattr(g, "vel_disp") else torch.ones(n)*100,
                    "half_mass_r": g.half_mass_r if hasattr(g, "half_mass_r") else torch.ones(n)*0.01,
                    "pos": g.pos if hasattr(g, "pos") else torch.zeros(n, 3),
                })
    return out

train_graphs = prepare(train_loader)
val_graphs = prepare(val_loader)
test_graphs = prepare(test_loader)
print("train graphs:", len(train_graphs), "| ctx dim:", train_graphs[0]["ctx"].shape)


In [ ]:
# CELL 7: Policy-gradient helpers (REINFORCE + learned baseline — honestly NOT PPO:
# one-step MDP => no GAE, no importance ratio; advantage = reward - baseline)
class ValueNet(nn.Module):
    def __init__(self, emb_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(emb_dim, hidden_dim), nn.LeakyReLU(0.1),
                                 nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(0.1),
                                 nn.Linear(hidden_dim, 1))
    def forward(self, ctx): return self.net(ctx).squeeze(-1)

def compute_advantages(rewards, values):
    return rewards - values.detach()

def bernoulli_entropy(p, eps=1e-6):
    # eps must be float32-representable: 1 - 1e-8 rounds to exactly 1.0 in
    # float32, making the upper clamp a no-op -> 0*log(0) = NaN for saturated
    # probs (sigmoid(x)>=17 saturates). Mirrors rls/policy_gradient.py and its
    # regression test test_entropy_and_logp_finite_at_saturated_probs.
    p = p.clamp(eps, 1.0 - eps)
    return -(p * torch.log(p) + (1 - p) * torch.log(1 - p))


In [ ]:
# CELL 8: Graph-physics terms (virial penalty) + reward
def graph_physics_terms(g, mask, G=4.302e-9):
    edge_index = g["edge_index"]; mask = mask.to(device)
    stellar = g["stellar_mass"].to(device); vd = g["vel_disp"].to(device); pos = g["pos"].to(device)
    deg = torch.zeros(stellar.shape[0], device=device)
    deg.index_add_(0, edge_index[0], torch.ones(edge_index.shape[1], device=device))
    deg.index_add_(0, edge_index[1], torch.ones(edge_index.shape[1], device=device))
    deg_ret = torch.zeros_like(deg)
    deg_ret.index_add_(0, edge_index[0, mask], torch.ones(mask.sum(), device=device))
    deg_ret.index_add_(0, edge_index[1, mask], torch.ones(mask.sum(), device=device))
    frac = deg_ret / deg.clamp(min=1)
    ke = 0.5 * torch.sum(frac * stellar * vd**2)
    u, v = edge_index[:, mask]
    r = torch.norm(pos[u] - pos[v], dim=1).clamp(min=1e-6)
    pe = G * torch.sum(stellar[u] * stellar[v] / r)
    return ke, pe

def compute_rewards(pred_pruned, pred_full, y, keep_ratio, target_sp, virial_pen, cfg, conn_ok):
    base = torch.sqrt(((pred_full - y)**2).mean() + 1e-8)
    prun = torch.sqrt(((pred_pruned - y)**2).mean())
    delta = (base - prun) / base
    sp = ((keep_ratio - target_sp)**2).clamp(max=1.0)
    cb = torch.tensor(1.0 if conn_ok else -1.0)
    return (cfg["w_acc"]*delta - cfg["w_sp"]*sp + cfg["w_conn"]*cb
            - cfg["w_virial"]*virial_pen)


In [ ]:
# CELL 9: GNN adapter — full and pruned predictions for a graph dict
def gnns_adapter(g, mask):
    def run(edge_index, edge_attr):
        d = pg.Data(x=g["x"], edge_index=edge_index, edge_attr=edge_attr)
        d.batch = torch.zeros(d.x.shape[0], dtype=torch.long, device=device)
        with torch.no_grad():
            pred, _ = gnn(d)      # forward returns (predictions, embeddings)
            return pred.view(-1)
    mask = mask.to(device)
    pred_full = run(g["edge_index"], g["edge_attr"])
    pred_pruned = run(g["edge_index"][:, mask], g["edge_attr"][mask])
    return pred_full, pred_pruned


## Load the already-trained policy (instead of retraining a third time)

In [ ]:
# CELL 10-LOAD (same as Notebook C — load the already-trained policy instead of
# training a third time)
import os as _os
rls = cfg["rls"]
policy = EdgePolicyNet(edge_dim=len(cfg["graph"]["edge_features"]),
                       node_emb_dim=cfg["model"]["output_dim"],
                       hidden_dim=rls["policy_hidden"]).to(device)
POLICY_INPUT = f"{INPUT}/policy.pt"
LOCAL_FALLBACK = "outputs/rls/policy.pt"
ckpt_path = POLICY_INPUT if _os.path.exists(POLICY_INPUT) else LOCAL_FALLBACK
if ckpt_path == LOCAL_FALLBACK:
    import datetime
    print("=" * 70)
    print(f"WARNING: policy loaded from LOCAL fallback {LOCAL_FALLBACK}")
    print(f"(file modified: {datetime.datetime.fromtimestamp(_os.path.getmtime(LOCAL_FALLBACK))})")
    print("A PRE-RL-FIX policy.pt is committed at this path. If Notebook B did NOT")
    print("just run in this session, these are STALE pre-fix weights and every")
    print("number below is invalid — upload the post-fix policy.pt to the")
    print("cosmicnet-data dataset so it loads from INPUT instead.")
    print("=" * 70)
policy.load_state_dict(torch.load(ckpt_path, map_location=device))
policy.eval()
print(f"loaded trained policy from {ckpt_path}")


The method: at inference, for EACH input graph, a copy of the policy takes K policy-gradient steps against a **label-free** reward (MC-dropout uncertainty reduction + sparsity + connectivity + virial), then emits the final mask. Labels are used ONLY for the final evaluation RMSE — never in the reward. FROZEN mode (K=0) is the ablation.

In [ ]:
# CELL D1: Label-free TTA reward + MC-dropout std helper
def mc_std(model, g, edge_index=None, edge_attr=None, n_samples=15):
    d = pg.Data(x=g["x"],
                edge_index=edge_index if edge_index is not None else g["edge_index"],
                edge_attr=edge_attr if edge_attr is not None else g["edge_attr"])
    d.batch = torch.zeros(d.x.shape[0], dtype=torch.long, device=device)
    out = model.predict_with_uncertainty(d, n_samples=n_samples)  # handles train/eval mode
    return out["std"].view(-1)

def label_free_reward(std_pruned, std_full, keep_ratio, target_sp, virial_pen, cfg, conn_ok):
    """NO labels anywhere — that is the whole point."""
    d_unc = (std_full - std_pruned) / (std_full + 1e-8)
    sp = min((keep_ratio - target_sp) ** 2, 1.0)
    cb = 1.0 if conn_ok else -1.0
    return (cfg["w_unc"] * d_unc - cfg["w_sp"] * sp + cfg["w_conn"] * cb
            - cfg.get("w_virial", 0.0) * virial_pen)


In [ ]:
# CELL D2: adapt_at_test_time — K policy-gradient steps per graph (deep copy; offline policy untouched)
import copy, time

def adapt_at_test_time(policy, g, cfg, init="offline", target_sp=None, log=False):
    target_sp = target_sp if target_sp is not None else cfg["tta_target_sparsity"]
    if init == "offline":
        pol = copy.deepcopy(policy).to(device)
    else:  # 'fresh' ablation
        torch.manual_seed(cfg.get("seed", 42))
        pol = EdgePolicyNet(edge_dim=g["edge_attr"].shape[1],
                            node_emb_dim=g["emb"].shape[1],
                            hidden_dim=cfg.get("policy_hidden", 64)).to(device)
    t_opt = torch.optim.Adam(pol.parameters(), lr=cfg["tta_lr"])
    g = {k: v.to(device) for k, v in g.items() if isinstance(v, torch.Tensor)}
    with torch.no_grad():
        std_full = mc_std(gnn, g, n_samples=cfg["tta_mc_samples"])   # cache once
    baseline, best_r, stall, hist = 0.0, -1e9, 0, []
    for step in range(cfg["tta_steps"]):
        probs = torch.sigmoid(pol(g["edge_attr"], g["emb"], g["edge_index"], g["ctx"])).squeeze(-1)
        action = torch.bernoulli(probs)
        # logp on the RAW sampled action; the reward mask below uses the SAME
        # action (floored/repaired) — like action-clipping in continuous control.
        logp = bernoulli_logp(probs, action).mean()
        ent = bernoulli_entropy(probs).mean()
        with torch.no_grad():
            m = repair_connectivity(g["edge_index"],
                                    apply_min_keep_floor(action.bool(), probs, cfg["min_keep_frac"]))
            std_pr = mc_std(gnn, g, g["edge_index"][:, m], g["edge_attr"][m],
                            n_samples=cfg["tta_mc_samples"])
            ke, pe = graph_physics_terms(g, m)
            vp = ((virial_ratio_pruned(ke, pe) - 1).clamp(min=0.0) ** 2) if cfg["w_virial"] > 0 else torch.zeros(1, device=device)
            deg = torch.zeros(g["x"].shape[0], device=device)
            deg.index_add_(0, g["edge_index"][0, m], torch.ones(m.sum(), device=device))
            deg.index_add_(0, g["edge_index"][1, m], torch.ones(m.sum(), device=device))
            conn_ok = bool((m.sum() > 0) and (deg >= 1).all())
            r = label_free_reward(std_pr, std_full, m.float().mean().item(), target_sp, vp, cfg, conn_ok)
        baseline = 0.9 * baseline + 0.1 * float(r)
        adv = float(r) - baseline
        loss = -(adv * logp) - cfg["entropy_coef"] * ent
        t_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(pol.parameters(), 1.0); t_opt.step()
        hist.append(float(r))
        stall = 0 if float(r) > best_r else stall + 1
        best_r = max(best_r, float(r))
        if stall >= cfg["tta_patience"]:
            break
        if log:
            print(f"  tta step {step}: r={float(r):.4f}")
    with torch.no_grad():
        p = torch.sigmoid(pol(g["edge_attr"], g["emb"], g["edge_index"], g["ctx"])).squeeze(-1)
        final = repair_connectivity(g["edge_index"], hard_mask(p, cfg["min_keep_frac"]))
    return final, {"reward_hist": hist, "steps_run": len(hist), "best_r": best_r}

def mask_to_pred(g, m):
    d = pg.Data(x=g["x"], edge_index=g["edge_index"][:, m], edge_attr=g["edge_attr"][m])
    d.batch = torch.zeros(d.x.shape[0], dtype=torch.long, device=device)
    with torch.no_grad():
        pred, _ = gnn(d)
        return pred.item()


## Fix: define `frozen_rmse` before the sanity assert in the next cell uses it

In [ ]:
# CELL D-FROZEN-CHECK: independent frozen-policy RMSE, computed once, used only
# to sanity-check CELL D3's K=0 branch below (see fix note at the top of this file).
# FIX: the original CELL D3 asserted against a variable `frozen_rmse` that was
# never defined anywhere -> guaranteed NameError on first run. This cell defines it.
frozen_preds, frozen_ys = [], []
with torch.no_grad():
    for g in test_graphs:
        p = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                 g["edge_index"].to(device), g["ctx"].to(device))).squeeze(-1)
        m = repair_connectivity(g["edge_index"].to(device), hard_mask(p, rls["min_keep_frac"]))
        gd = {k: v.to(device) for k, v in g.items() if isinstance(v, torch.Tensor)}
        frozen_preds.append(mask_to_pred(gd, m))
        frozen_ys.append(float(g["y"].view(-1)[0]))
frozen_rmse = float(np.sqrt(((np.array(frozen_preds) - np.array(frozen_ys)) ** 2).mean()))
print("independent frozen_rmse check:", frozen_rmse)


In [ ]:
# CELL D3: In-domain TTA — frozen (K=0) vs TTA (K ablation) on the TNG test set
# Uses test_graphs (from Notebook B cell 6: dicts with x/edge_index/edge_attr/y/ctx/emb/pos/...)
tta_rows = []
for K in [0, 5, 10, 20]:
    preds, fulls, ys, keeps, steps, secs, hists = [], [], [], [], [], [], []
    for g in test_graphs:
        gd = {k: v.to(device) for k, v in g.items() if isinstance(v, torch.Tensor)}
        t0 = time.time()
        if K == 0:  # FROZEN mode
            with torch.no_grad():
                p = torch.sigmoid(policy(gd["edge_attr"], gd["emb"], gd["edge_index"], gd["ctx"])).squeeze(-1)
                m = repair_connectivity(gd["edge_index"], hard_mask(p, rls["min_keep_frac"]))
            info = {"steps_run": 0, "reward_hist": []}
        else:
            m, info = adapt_at_test_time(policy, g, rls, init="offline")
        secs.append(time.time() - t0)
        preds.append(mask_to_pred(gd, m)); ys.append(float(gd["y"].view(-1)[0]))
        with torch.no_grad():
            dfull = pg.Data(x=gd["x"], edge_index=gd["edge_index"], edge_attr=gd["edge_attr"])
            dfull.batch = torch.zeros(dfull.x.shape[0], dtype=torch.long, device=device)
            pf, _ = gnn(dfull)
        fulls.append(pf.item())
        keeps.append(float(m.float().mean())); steps.append(info["steps_run"]); hists.append(info["reward_hist"])
    preds, fulls, ys = np.array(preds), np.array(fulls), np.array(ys)
    row = {"mode": "frozen" if K == 0 else "tta", "K": K,
           "rmse": float(np.sqrt(((preds - ys) ** 2).mean())),
           "fidelity": float(np.corrcoef(preds, fulls)[0, 1]),
           "keep_frac": float(np.mean(keeps)), "mean_steps": float(np.mean(steps)),
           "mean_time_s": float(np.mean(secs))}
    tta_rows.append(row)
    print(row)
tta_df = pd.DataFrame(tta_rows)
tta_df.to_csv("outputs/rls/tta_indomain.csv", index=False)
assert abs(tta_df.loc[tta_df.K == 0, "rmse"].iloc[0] - frozen_rmse) < 1e-6, \
    "K=0 must reproduce the frozen policy EXACTLY (same policy, same mask, same repair)"


In [ ]:
# CELL D4: OOD TTA — TNG-trained policy on CAMELS, frozen vs TTA (THE headline)
# Requires REAL CAMELS HDF5 (synthetic fallback is NOT publishable).
# Adaptation needs NO labels — that is why TTA works here and the frozen policy cannot.
cfg2 = yaml.safe_load(open("config/config.yaml"))
cfg2["data"]["source"] = "camels"
cfg2["data"]["camels"] = {"suite": "IllustrisTNG", "simulation": "LH_0",
                          "cache_dir": "/kaggle/working/camels_cache"}
cfg2["data"]["num_workers"] = 0
try:
    loader2 = get_loader(cfg2)
    halos2 = loader2.load()
    assert getattr(loader2, "used_synthetic_fallback", False) is False, \
        "synthetic CAMELS fallback is not publishable — patch camels_loader.py " \
        "(plan Task 14 Step 6) and cache the real HDF5 first"
    gb2 = GraphBuilder(cfg2)
    graphs2 = gb2.build_graphs(halos2[:100])
    from torch_geometric.data import Batch as PyGBatch
    camels_graphs = []
    with torch.no_grad():
        for g in graphs2:
            g = g.to(device)
            gb = PyGBatch.from_data_list([g])
            emb = gnn.get_embeddings(gb, embedding_point="pre_pooling")
            ctx = global_mean_pool(emb, gb.batch)
            n = g.x.shape[0]
            camels_graphs.append({
                "x": g.x, "edge_index": g.edge_index, "edge_attr": g.edge_attr, "y": g.y,
                "ctx": ctx[0], "emb": emb,
                "stellar_mass": g.stellar_mass if hasattr(g, "stellar_mass") else torch.ones(n, device=device) * 1e10,
                "vel_disp": g.vel_disp if hasattr(g, "vel_disp") else torch.ones(n, device=device) * 100,
                "half_mass_r": g.half_mass_r if hasattr(g, "half_mass_r") else torch.ones(n, device=device) * 0.01,
                "pos": g.pos if hasattr(g, "pos") else torch.zeros(n, 3, device=device),
            })
    ood_rows = []
    for mode in ["frozen", "tta"]:
        preds, ys = [], []
        for g in camels_graphs:
            gd = {k: v for k, v in g.items() if isinstance(v, torch.Tensor)}
            if mode == "frozen":
                with torch.no_grad():
                    p = torch.sigmoid(policy(gd["edge_attr"], gd["emb"], gd["edge_index"], gd["ctx"])).squeeze(-1)
                    m = repair_connectivity(gd["edge_index"], hard_mask(p, rls["min_keep_frac"]))
            else:
                m, info = adapt_at_test_time(policy, g, rls, init="offline")
            preds.append(mask_to_pred(gd, m)); ys.append(float(gd["y"].view(-1)[0]))
        preds, ys = np.array(preds), np.array(ys)
        row = {"mode": mode, "rmse": float(np.sqrt(((preds - ys) ** 2).mean())),
               "r2": float(1 - ((preds - ys) ** 2).sum() / ((ys - ys.mean()) ** 2).sum())}
        ood_rows.append(row); print("CAMELS", row)
    pd.DataFrame(ood_rows).to_csv("outputs/rls/tta_camels.csv", index=False)
    delta = ood_rows[1]["rmse"] - ood_rows[0]["rmse"]
    print(f"HEADLINE: CAMELS OOD RMSE frozen={ood_rows[0]['rmse']:.4f} vs TTA={ood_rows[1]['rmse']:.4f} "
          f"(TTA change: {delta:+.4f} dex). This is the ABSOLUTE gap — a 'fraction of "
          "degradation recovered' claim also needs the in-domain frozen RMSE as reference.")
except Exception as e:
    print("CAMELS OOD skipped/failed:", e)
    print("If this is the synthetic fallback, do NOT put these numbers in the paper.")


In [ ]:
# CELL D5: TTA plots + reward trajectories + save
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(tta_df.K.astype(str), tta_df.rmse, color=["gray"] + ["steelblue"] * 3)
axes[0].set_xlabel("TTA steps K (0 = frozen)"); axes[0].set_ylabel("test RMSE (dex)")
axes[0].set_title("In-domain: TTA vs frozen")
axes[1].bar(tta_df.K.astype(str), tta_df.mean_time_s, color=["gray"] + ["darkorange"] * 3)
axes[1].set_xlabel("TTA steps K"); axes[1].set_ylabel("adaptation time (s/graph)")
axes[1].set_title("TTA cost")
for h in hists[:5]:
    if h: axes[2].plot(h, alpha=0.7)
axes[2].set_xlabel("TTA step"); axes[2].set_ylabel("label-free reward")
axes[2].set_title("Reward trajectories (5 graphs)")
fig.tight_layout(); fig.savefig("outputs/rls/tta_results.png", dpi=200)
shutil.make_archive("/kaggle/working/rls_outputs", "zip", "outputs/rls")
print("saved outputs/rls/tta_*.csv + tta_results.png — download rls_outputs.zip")


> **TTA pitfalls (plan Part 4):** if TTA's test RMSE is WORSE than frozen while Δunc is positive, the uncertainty reward is being gamed → raise `w_virial`/`w_conn` or lower K. Tune (K, tta_lr, w_unc) on the VAL split only. K=0 must reproduce the frozen numbers exactly.